In [1]:
# CELDA 1 — Setup del notebook de Feature Engineering
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import unicodedata
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

# Paths
PROJECT_PATH = '/content/drive/MyDrive/HortifrutCostosImport'
PROCESSED_PATH = f'{PROJECT_PATH}/data/processed'
FIGURES_PATH = f'{PROJECT_PATH}/reports/figures'

# Cargar dataset del notebook anterior
df = pd.read_parquet(f'{PROCESSED_PATH}/dataset_eda.parquet')
print(f"Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Operaciones únicas: {df['Nro. Ope.'].nunique():,}")
print(f"Rango temporal: {df['Fecha_Imputada'].min().strftime('%Y-%m-%d')} → {df['Fecha_Imputada'].max().strftime('%Y-%m-%d')}")

Mounted at /content/drive
Dataset cargado: 14,705 filas × 56 columnas
Operaciones únicas: 1,252
Rango temporal: 2020-07-01 → 2026-07-01


In [2]:
# CELDA 2 — Excluir notas de crédito (montos negativos y ceros)
n_inicial = len(df)

# Diagnóstico antes del descarte
neg = df[df['Importe_Total_PEN'] < 0]
ceros = df[df['Importe_Total_PEN'] == 0]
nans = df[df['Importe_Total_PEN'].isna()]

print(f"📊 Diagnóstico antes del descarte:")
print(f"  Total filas:              {n_inicial:,}")
print(f"  Filas negativas (NC):     {len(neg):,}  (Monto: S/ {neg['Importe_Total_PEN'].sum():,.0f})")
print(f"  Filas con monto cero:     {len(ceros):,}")
print(f"  Filas con monto NaN:      {len(nans):,}")

# Aplicar filtro: solo facturas con monto positivo
df = df[df['Importe_Total_PEN'] > 0].copy().reset_index(drop=True)

print(f"\n✓ Dataset filtrado: {len(df):,} filas (se excluyeron {n_inicial - len(df):,})")
print(f"  Porcentaje conservado: {len(df)/n_inicial*100:.2f}%")

📊 Diagnóstico antes del descarte:
  Total filas:              14,705
  Filas negativas (NC):     32  (Monto: S/ -115,741)
  Filas con monto cero:     13
  Filas con monto NaN:      0

✓ Dataset filtrado: 14,660 filas (se excluyeron 45)
  Porcentaje conservado: 99.69%


In [3]:
# CELDA 3 — Features temporales a partir de Fecha_Imputada
df['año']               = df['Fecha_Imputada'].dt.year
df['mes']               = df['Fecha_Imputada'].dt.month
df['trimestre']         = df['Fecha_Imputada'].dt.quarter
df['semana_año']        = df['Fecha_Imputada'].dt.isocalendar().week.astype(int)
df['dia_semana']        = df['Fecha_Imputada'].dt.dayofweek  # 0=lunes, 6=domingo

# Tiempo lineal desde el inicio del histórico (útil para capturar tendencias)
fecha_inicio = df['Fecha_Imputada'].min()
df['dias_desde_inicio'] = (df['Fecha_Imputada'] - fecha_inicio).dt.days

# Flag de temporada alta de berries en Perú (agosto - diciembre)
df['es_temporada_alta'] = df['mes'].isin([8, 9, 10, 11, 12]).astype(int)

print("Features temporales creadas:")
nuevas = ['año', 'mes', 'trimestre', 'semana_año', 'dia_semana', 'dias_desde_inicio', 'es_temporada_alta']
print(df[nuevas].describe().round(1).to_string())

print(f"\nDistribución por mes (cuántas facturas por mes calendario):")
print(df['mes'].value_counts().sort_index().to_string())

print(f"\nProporción en temporada alta vs baja:")
print(df['es_temporada_alta'].value_counts(normalize=True).round(3).to_string())

Features temporales creadas:
           año      mes  trimestre  semana_año  dia_semana  dias_desde_inicio  es_temporada_alta
count  14660.0  14660.0    14660.0     14660.0     14660.0            14660.0            14660.0
mean    2022.8      6.7        2.7        26.8         2.3             1027.5                0.3
std        1.6      3.2        1.1        13.7         1.5              599.4                0.5
min     2020.0      1.0        1.0         1.0         0.0                0.0                0.0
25%     2022.0      4.0        2.0        16.0         1.0              572.0                0.0
50%     2022.0      7.0        3.0        26.0         2.0              852.0                0.0
75%     2024.0      9.0        3.0        38.0         4.0             1617.0                1.0
max     2026.0     12.0        4.0        52.0         6.0             2191.0                1.0

Distribución por mes (cuántas facturas por mes calendario):
mes
1     1009
2     1209
3     1104


In [4]:
# CELDA 4 — Consolidar duplicados ortográficos en categóricas
def normalizar_categorica(s):
    """Mayúsculas, sin tildes, sin puntos, espacios colapsados."""
    if pd.isna(s):
        return s
    s = str(s).upper().strip()
    # Sacar tildes (NFD descompone tilde+letra, después filtramos las tildes)
    s = ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')
    # Sacar puntos y comas
    s = s.replace('.', '').replace(',', '')
    # Colapsar espacios múltiples
    s = ' '.join(s.split())
    return s

# Columnas a normalizar (ajustá los nombres si en tu df aparecen distintos)
columnas_a_consolidar = ['Proveedor Principal', 'Producto', 'AGENCIA DE ADUANA', 'ACREEDOR', 'Proveedor']

print(f"{'Columna':<25} {'Antes':>10} {'Después':>10} {'Reducción':>12}")
print("-" * 60)

for col in columnas_a_consolidar:
    if col not in df.columns:
        print(f"{col:<25}   (no existe en el dataset)")
        continue
    antes = df[col].nunique()
    df[f'{col}_norm'] = df[col].apply(normalizar_categorica)
    despues = df[f'{col}_norm'].nunique()
    reduccion = antes - despues
    pct = reduccion / antes * 100 if antes else 0
    print(f"{col:<25} {antes:>10,} {despues:>10,} {reduccion:>9,} ({pct:.1f}%)")

print(f"\nDataset ahora tiene {df.shape[1]} columnas")

Columna                        Antes    Después    Reducción
------------------------------------------------------------
Proveedor Principal              164        155         9 (5.5%)
Producto                         379        364        15 (4.0%)
AGENCIA DE ADUANA                 23         21         2 (8.7%)
ACREEDOR                          77         66        11 (14.3%)
Proveedor                        175        148        27 (15.4%)

Dataset ahora tiene 68 columnas


In [5]:
# CELDA 5 — Agregar flag fecha_original (1 = venía del archivo, 0 = fue imputada al 1-jul)
df['fecha_original'] = df['Fecha de Emisión Doc'].notna().astype(int)

print(f"Distribución del flag fecha_original:")
print(df['fecha_original'].value_counts().to_string())
print(f"\n  Filas con fecha REAL del archivo:  {(df['fecha_original']==1).sum():,} ({(df['fecha_original']==1).mean()*100:.1f}%)")
print(f"  Filas con fecha imputada (1-jul):  {(df['fecha_original']==0).sum():,} ({(df['fecha_original']==0).mean()*100:.1f}%)")

# Verificación: mirar la distribución de mes SOLO para fechas reales
print(f"\n📅 Distribución por mes — SOLO fechas reales (sin el artefacto):")
print(df[df['fecha_original']==1]['mes'].value_counts().sort_index().to_string())

Distribución del flag fecha_original:
fecha_original
1    10009
0     4651

  Filas con fecha REAL del archivo:  10,009 (68.3%)
  Filas con fecha imputada (1-jul):  4,651 (31.7%)

📅 Distribución por mes — SOLO fechas reales (sin el artefacto):
mes
1     1009
2     1209
3     1104
4      460
5      333
6      455
7      692
8      639
9      928
10    1040
11    1114
12    1026


In [6]:
# CELDA 6 — Features derivadas del despacho
# Flag de peso disponible (importante porque solo está en 2025-26)
df['peso_disponible'] = df['Peso Bruto (kg)'].notna().astype(int)

# Densidad de bultos por contenedor
df['bultos_por_contenedor'] = np.where(
    (df['Cantidad de Contenedores'] > 0) & df['Cantidad de Bultos (BULKS)'].notna(),
    df['Cantidad de Bultos (BULKS)'] / df['Cantidad de Contenedores'],
    np.nan
)

# Peso por contenedor (solo cuando ambos existen)
df['peso_por_contenedor'] = np.where(
    (df['Cantidad de Contenedores'] > 0) & df['Peso Bruto (kg)'].notna(),
    df['Peso Bruto (kg)'] / df['Cantidad de Contenedores'],
    np.nan
)

# Flag binario de proyecto/programa
df['tiene_proyecto'] = (
    df['Proyecto'].notna() & (df['Proyecto'].astype(str).str.strip() != '-')
).astype(int)

# Agrupar Incoterm en familias (E, F, C, D)
def incoterm_familia(s):
    if pd.isna(s):
        return 'DESCONOCIDO'
    s = str(s).upper().strip()
    if s.startswith('E'):   return 'GRUPO_E'   # EXW
    if s.startswith('F'):   return 'GRUPO_F'   # FOB, FCA, FAS
    if s.startswith('C'):   return 'GRUPO_C'   # CFR, CIF, CPT, CIP
    if s.startswith('D'):   return 'GRUPO_D'   # DAP, DAT, DDP
    return 'OTRO'

df['incoterm_familia'] = df['Incoterm'].apply(incoterm_familia)

# Resumen
print("Features derivadas creadas:\n")

print("• peso_disponible (1 = tiene peso bruto):")
print(df['peso_disponible'].value_counts(normalize=True).round(3).to_string())

print("\n• bultos_por_contenedor:")
print(df['bultos_por_contenedor'].describe().round(1).to_string())

print("\n• tiene_proyecto:")
print(df['tiene_proyecto'].value_counts(normalize=True).round(3).to_string())

print("\n• incoterm_familia:")
print(df['incoterm_familia'].value_counts().to_string())

print(f"\nDataset ahora tiene {df.shape[1]} columnas")

Features derivadas creadas:

• peso_disponible (1 = tiene peso bruto):
peso_disponible
0    0.905
1    0.095

• bultos_por_contenedor:
count    8732.0
mean      131.5
std       247.2
min         0.0
25%        20.0
50%        22.0
75%        24.0
max      1350.0

• tiene_proyecto:
tiene_proyecto
0    0.575
1    0.425

• incoterm_familia:
incoterm_familia
GRUPO_C        7936
GRUPO_E        4372
GRUPO_F        1939
GRUPO_D         264
DESCONOCIDO     149

Dataset ahora tiene 74 columnas


In [7]:
# CELDA 7 — Target encoding histórico (tarifa histórica derivada)

# Asegurar que las columnas categóricas no tengan NaN (los reemplazamos por 'DESCONOCIDO')
df['Proveedor_norm'] = df['Proveedor_norm'].fillna('DESCONOCIDO')
df['ACREEDOR_norm']  = df['ACREEDOR_norm'].fillna('DESCONOCIDO')

# Ordenar por fecha (crítico para que el "expanding" funcione cronológicamente)
df = df.sort_values('Fecha_Imputada').reset_index(drop=True)

# Nivel 1 — Tarifa histórica por (Proveedor de servicio × Concepto Canónico)
# Para cada fila: mediana de las facturas anteriores del mismo proveedor × concepto
df['tarifa_hist_prov_concepto'] = (
    df.groupby(['Proveedor_norm', 'Concepto Canónico'])['Importe_Total_PEN']
      .transform(lambda x: x.expanding().median().shift(1))
)

# Nivel 2 — Fallback por Concepto Canónico (cuando es un proveedor nuevo)
df['tarifa_hist_concepto'] = (
    df.groupby('Concepto Canónico')['Importe_Total_PEN']
      .transform(lambda x: x.expanding().median().shift(1))
)

# Nivel 3 — Fallback global (cuando es un concepto nuevo)
df['tarifa_hist_global'] = df['Importe_Total_PEN'].expanding().median().shift(1)

# Tarifa histórica final con cascada de fallbacks
df['tarifa_historica'] = (
    df['tarifa_hist_prov_concepto']
      .fillna(df['tarifa_hist_concepto'])
      .fillna(df['tarifa_hist_global'])
)

# Frequency encoding: cuántas veces vimos a este proveedor antes (también con shift)
df['proveedor_frecuencia'] = (
    df.groupby('Proveedor_norm').cumcount()
)

# Diagnóstico
print(f"📊 Cobertura de la tarifa histórica:")
print(f"  Filas con tarifa_hist_prov_concepto (nivel 1): {df['tarifa_hist_prov_concepto'].notna().sum():,} ({df['tarifa_hist_prov_concepto'].notna().mean()*100:.1f}%)")
print(f"  Filas con tarifa_hist_concepto (nivel 2):      {df['tarifa_hist_concepto'].notna().sum():,} ({df['tarifa_hist_concepto'].notna().mean()*100:.1f}%)")
print(f"  Filas con tarifa_historica (cualquier nivel):  {df['tarifa_historica'].notna().sum():,} ({df['tarifa_historica'].notna().mean()*100:.1f}%)")

print(f"\n💰 Distribución de tarifa_historica:")
print(df['tarifa_historica'].describe().round(0).to_string())

# Comparar tarifa histórica vs valor real (esto NO es para el modelo, es solo diagnóstico)
correlacion = df[['tarifa_historica', 'Importe_Total_PEN']].dropna().corr().iloc[0, 1]
print(f"\n🔍 Correlación entre tarifa_historica y Importe_Total_PEN: {correlacion:.3f}")
print("   (esperamos correlación alta: 0.5-0.9 es buena señal de feature predictivo)")

📊 Cobertura de la tarifa histórica:
  Filas con tarifa_hist_prov_concepto (nivel 1): 14,295 (97.5%)
  Filas con tarifa_hist_concepto (nivel 2):      14,647 (99.9%)
  Filas con tarifa_historica (cualquier nivel):  14,659 (100.0%)

💰 Distribución de tarifa_historica:
count     14659.0
mean       3872.0
std        5602.0
min           6.0
25%         331.0
50%        1349.0
75%        3937.0
max      101331.0

🔍 Correlación entre tarifa_historica y Importe_Total_PEN: 0.355
   (esperamos correlación alta: 0.5-0.9 es buena señal de feature predictivo)


In [8]:
# CELDA 8 — Split train/test temporal
cutoff_date = pd.Timestamp('2025-01-01')

train = df[df['Fecha_Imputada'] < cutoff_date].copy().reset_index(drop=True)
test  = df[df['Fecha_Imputada'] >= cutoff_date].copy().reset_index(drop=True)

print(f"📅 División temporal — corte: {cutoff_date.date()}")
print(f"\nTRAIN: {len(train):,} filas ({len(train)/len(df)*100:.1f}%)")
print(f"  Rango: {train['Fecha_Imputada'].min().date()}  →  {train['Fecha_Imputada'].max().date()}")
print(f"  Operaciones únicas: {train['Nro. Ope.'].nunique():,}")
print(f"  Importe Total (S/): mediana={train['Importe_Total_PEN'].median():,.0f}  promedio={train['Importe_Total_PEN'].mean():,.0f}")

print(f"\nTEST:  {len(test):,} filas ({len(test)/len(df)*100:.1f}%)")
print(f"  Rango: {test['Fecha_Imputada'].min().date()}  →  {test['Fecha_Imputada'].max().date()}")
print(f"  Operaciones únicas: {test['Nro. Ope.'].nunique():,}")
print(f"  Importe Total (S/): mediana={test['Importe_Total_PEN'].median():,.0f}  promedio={test['Importe_Total_PEN'].mean():,.0f}")

# Validación: operaciones que aparecen en ambos sets
ops_train = set(train['Nro. Ope.'].unique())
ops_test = set(test['Nro. Ope.'].unique())
overlap = ops_train & ops_test
print(f"\n🔍 Operaciones que aparecen en AMBOS sets: {len(overlap)}")
print(f"   (esto es esperable: una operación abierta en 2024 puede tener facturas que llegan en 2025)")
print(f"   En producción, el modelo predice factura por factura, así que esto NO es leakage.")

# Distribución de conceptos canónicos en train vs test
print(f"\n📊 Distribución de Concepto Canónico en train vs test:")
dist = pd.DataFrame({
    'train_%': (train['Concepto Canónico'].value_counts(normalize=True) * 100).round(1),
    'test_%':  (test['Concepto Canónico'].value_counts(normalize=True) * 100).round(1)
}).fillna(0)
print(dist.to_string())

📅 División temporal — corte: 2025-01-01

TRAIN: 11,181 filas (76.3%)
  Rango: 2020-07-01  →  2024-12-31
  Operaciones únicas: 1,045
  Importe Total (S/): mediana=1,096  promedio=9,038

TEST:  3,479 filas (23.7%)
  Rango: 2025-01-01  →  2026-07-01
  Operaciones únicas: 346
  Importe Total (S/): mediana=1,126  promedio=4,564

🔍 Operaciones que aparecen en AMBOS sets: 139
   (esto es esperable: una operación abierta en 2024 puede tener facturas que llegan en 2025)
   En producción, el modelo predice factura por factura, así que esto NO es leakage.

📊 Distribución de Concepto Canónico en train vs test:
                           train_%  test_%
Concepto Canónico                         
AGENCIAMIENTO_ADUANA           6.6     9.0
DERECHOS_IMPUESTOS            16.9     7.3
DESCARGA                      11.9    11.3
FITOSANITARIOS_SENASA          8.7     4.6
FLETE_INTERNACIONAL            7.1     7.3
HANDLING_PUERTO               10.1    12.0
INSPECCION_VERIFICACION       11.4    17.2
OTROS  

In [9]:
# CELDA 9 — Guardar dataset modelable final
train.to_parquet(f'{PROCESSED_PATH}/dataset_modelable_train.parquet', index=False)
test.to_parquet(f'{PROCESSED_PATH}/dataset_modelable_test.parquet', index=False)

print(f"✓ Guardado dataset_modelable_train.parquet")
print(f"  Tamaño: {os.path.getsize(f'{PROCESSED_PATH}/dataset_modelable_train.parquet') / 1024:.1f} KB")
print(f"  Filas:  {len(train):,}")

print(f"\n✓ Guardado dataset_modelable_test.parquet")
print(f"  Tamaño: {os.path.getsize(f'{PROCESSED_PATH}/dataset_modelable_test.parquet') / 1024:.1f} KB")
print(f"  Filas:  {len(test):,}")

# Resumen de columnas (lo que tenemos disponible para modelar)
print(f"\n📋 Total de columnas disponibles para modelar: {df.shape[1]}")
print(f"\nColumnas creadas durante Fase 3 (feature engineering):")
features_nuevas = ['año', 'mes', 'trimestre', 'semana_año', 'dia_semana', 'dias_desde_inicio',
                   'es_temporada_alta', 'fecha_original', 'peso_disponible',
                   'bultos_por_contenedor', 'peso_por_contenedor', 'tiene_proyecto',
                   'incoterm_familia', 'tarifa_hist_prov_concepto', 'tarifa_hist_concepto',
                   'tarifa_hist_global', 'tarifa_historica', 'proveedor_frecuencia']
for c in features_nuevas:
    if c in df.columns:
        print(f"  • {c}")

✓ Guardado dataset_modelable_train.parquet
  Tamaño: 876.6 KB
  Filas:  11,181

✓ Guardado dataset_modelable_test.parquet
  Tamaño: 289.5 KB
  Filas:  3,479

📋 Total de columnas disponibles para modelar: 79

Columnas creadas durante Fase 3 (feature engineering):
  • año
  • mes
  • trimestre
  • semana_año
  • dia_semana
  • dias_desde_inicio
  • es_temporada_alta
  • fecha_original
  • peso_disponible
  • bultos_por_contenedor
  • peso_por_contenedor
  • tiene_proyecto
  • incoterm_familia
  • tarifa_hist_prov_concepto
  • tarifa_hist_concepto
  • tarifa_hist_global
  • tarifa_historica
  • proveedor_frecuencia
